# Concat & Reshape Normal Task Features → Tidy Modelling CSV

**Input files (12 CSVs):**
- `resting_mav_normal - C3/C4/CZ.csv`
- `resting_var_normal - C3/C4/Cz.csv`
- `thinking_mav_normal - C3/C4/Cz.csv`
- `thinking_var_normal - C3/C4/Cz.csv`

**Raw layout per file:**  
Row 1 = subband name headers (High_Beta / Low_Beta / Mu)  
Row 2 = column labels (ID, scenario1 … scenario9, repeated ×3)  
Row 3+ = one row per subject, values repeated in 3 blocks (one per subband)

**Output:** `normal_task_features_tidy.csv`  
One row = one (subject × task × channel × subband × scenario) observation.  
Columns: `subject_id`, `task`, `channel`, `subband`, `scenario`, `mav`, `variance`

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
import re
import warnings
warnings.filterwarnings('ignore')

# ── Paths ─────────────────────────────────────────────────────────────────────
ROOT       = Path('.').resolve()
INPUT_CSV  = ROOT / 'normal_task_features_concat_only.csv'   # already-concatenated raw CSV
OUTPUT_CSV = ROOT / 'normal_task_features_Formatted.csv'

print(f'Working dir : {ROOT}')
print(f'Input  CSV  : {INPUT_CSV}')
print(f'Output CSV  : {OUTPUT_CSV}')

In [ ]:
# ── Load the raw concatenated CSV ─────────────────────────────────────────────
raw = pd.read_csv(
    INPUT_CSV,
    dtype=str,
    keep_default_na=False,
    na_filter=False,
    encoding='utf-8-sig',
)

print(f'Raw shape : {raw.shape}')
print(f'Columns   : {list(raw.columns)}')
print(f'Unique source files ({raw["source_file"].nunique()}):')
for f in sorted(raw['source_file'].unique()):
    print(f'  {f}')

In [ ]:
# ── Column layout (fixed for all 12 source files) ─────────────────────────────
#
# Each source CSV has 3 subband blocks, each 10 cols wide:
#   Block 1  → High_Beta : col_2  = subject_id, col_3  … col_11 = scenario1…9
#   Block 2  → Low_Beta  : col_12 = subject_id, col_13 … col_21 = scenario1…9
#   Block 3  → Mu        : col_22 = subject_id, col_23 … col_31 = scenario1…9
#
# Row 1 (source_row == '1') : subband name labels  → skip
# Row 2 (source_row == '2') : ID / scenario labels → skip
# Row 3+                    : actual data

SUBBAND_BLOCKS = {
    'High_Beta': ('col_2',  [f'col_{i}' for i in range(3,  12)]),   # col_3  … col_11
    'Low_Beta':  ('col_12', [f'col_{i}' for i in range(13, 22)]),   # col_13 … col_21
    'Mu':        ('col_22', [f'col_{i}' for i in range(23, 32)]),   # col_23 … col_31
}
SCENARIOS = [f'scenario{i}' for i in range(1, 10)]

# ── Drop the two header rows from every source file ───────────────────────────
data_rows = raw[~raw['source_row'].isin(['1', '2'])].copy()
print(f'Data rows after removing header rows: {len(data_rows):,}')

# ── Parse each block into long format ─────────────────────────────────────────
frames = []

for subband, (id_col, val_cols) in SUBBAND_BLOCKS.items():
    block = data_rows[['task', 'feature_type', 'channel', id_col] + val_cols].copy()
    block.columns = ['task', 'feature_type', 'channel', 'subject_id'] + SCENARIOS

    # Melt: one row per scenario
    melted = block.melt(
        id_vars=['task', 'feature_type', 'channel', 'subject_id'],
        value_vars=SCENARIOS,
        var_name='scenario',
        value_name='value',
    )
    melted['subband'] = subband
    frames.append(melted)

long_df = pd.concat(frames, ignore_index=True)
print(f'Long format rows: {len(long_df):,}')

In [ ]:
# ── Clean up ──────────────────────────────────────────────────────────────────

# Remove empty subject IDs (artifact of header rows not fully filtered)
long_df = long_df[long_df['subject_id'].str.strip().ne('')].copy()

# Remove rows where value is empty string
long_df = long_df[long_df['value'].str.strip().ne('')].copy()

# Convert value to float
long_df['value'] = pd.to_numeric(long_df['value'], errors='coerce')

# Drop any rows that failed numeric conversion
bad = long_df['value'].isna().sum()
if bad > 0:
    print(f'Dropping {bad} rows with non-numeric value')
    long_df = long_df.dropna(subset=['value'])

# Normalise channel name to uppercase
long_df['channel'] = long_df['channel'].str.upper().str.strip()

# Extract scenario number as integer
long_df['scenario_num'] = long_df['scenario'].str.extract(r'(\d+)').astype(int)

print(f'Rows after cleaning: {len(long_df):,}')
print('Tasks   :', long_df['task'].unique().tolist())
print('Channels:', sorted(long_df['channel'].unique().tolist()))
print('Subbands:', long_df['subband'].unique().tolist())
print('Features:', long_df['feature_type'].unique().tolist())
long_df.head()

In [ ]:
# ── Pivot: mav & variance become separate columns ────────────────────────────
#
# Each (subject, task, channel, subband, scenario) should have exactly
# one mav value and one variance value.

tidy = long_df.pivot_table(
    index=['subject_id', 'task', 'channel', 'subband', 'scenario', 'scenario_num'],
    columns='feature_type',
    values='value',
    aggfunc='mean',      # in case of duplicates, take mean
).reset_index()

tidy.columns.name = None   # remove axis name

# Rename columns neatly
col_rename = {}
if 'mav'      in tidy.columns: col_rename['mav']      = 'mav'
if 'variance' in tidy.columns: col_rename['variance']  = 'variance'

# Sort for readability
tidy = tidy.sort_values(
    ['subject_id', 'task', 'channel', 'subband', 'scenario_num']
).reset_index(drop=True)

print(f'Tidy shape: {tidy.shape}')
print(f'Columns   : {list(tidy.columns)}')
tidy.head(12)

In [ ]:
# ── Validation ────────────────────────────────────────────────────────────────
print('=== Tidy Dataset Validation ===')
print(f'  Rows         : {len(tidy):,}')
print(f'  Columns      : {list(tidy.columns)}')
print(f'  NaN in mav   : {tidy["mav"].isna().sum()}')
print(f'  NaN in var   : {tidy["variance"].isna().sum()}')
print(f'  Subjects     : {tidy["subject_id"].nunique()}')
print(f'  Tasks        : {tidy["task"].unique().tolist()}')
print(f'  Channels     : {sorted(tidy["channel"].unique().tolist())}')
print(f'  Subbands     : {tidy["subband"].unique().tolist()}')
print(f'  Scenarios    : {sorted(tidy["scenario_num"].unique().tolist())}')

print('\nRows per (task × channel × subband):')
print(
    tidy.groupby(['task','channel','subband']).size()
    .unstack(fill_value=0).to_string()
)

In [ ]:
# ── Build WIDE format for modelling ──────────────────────────────────────────
#
# One row = one (subject_id, scenario_num)
# Columns = task_channel_subband_mav  /  task_channel_subband_variance
# This is the format ready for sklearn / XGBoost / etc.

tidy['col_prefix'] = (
    tidy['task'] + '_' + tidy['channel'] + '_' + tidy['subband']
)

parts = []
for feat in ['mav', 'variance']:
    if feat not in tidy.columns:
        continue
    pivot = tidy.pivot_table(
        index=['subject_id', 'scenario', 'scenario_num'],
        columns='col_prefix',
        values=feat,
        aggfunc='mean',
    )
    pivot.columns = [f'{col}_{feat}' for col in pivot.columns]
    parts.append(pivot)

wide = pd.concat(parts, axis=1).reset_index()
wide = wide.sort_values(['subject_id', 'scenario_num']).reset_index(drop=True)

# Fill any NaN (subject × scenario combos with no data) with column median
feat_cols_wide = [c for c in wide.columns if c not in ['subject_id','scenario','scenario_num']]
wide[feat_cols_wide] = wide[feat_cols_wide].fillna(wide[feat_cols_wide].median())

print(f'Wide format shape: {wide.shape}')
print(f'Feature columns  : {len(feat_cols_wide)}')
wide.head(6)

In [ ]:
# ── Save both formats ─────────────────────────────────────────────────────────

# 1. TIDY (long) — best for EDA and grouped analysis
tidy_out = OUTPUT_CSV
tidy.drop(columns=['col_prefix'], errors='ignore').to_csv(tidy_out, index=False)
print(f'Saved TIDY   → {tidy_out}  ({len(tidy):,} rows × {tidy.shape[1]} cols)')

# 2. WIDE — ready for sklearn / modelling
wide_out = OUTPUT_CSV.parent / 'normal_task_features_wide.csv'
wide.to_csv(wide_out, index=False)
print(f'Saved WIDE   → {wide_out}  ({len(wide):,} rows × {wide.shape[1]} cols)')

# Final NaN check
print(f'\nNaN in tidy  mav      : {tidy["mav"].isna().sum()}')
print(f'NaN in tidy  variance : {tidy["variance"].isna().sum()}')
print(f'NaN in wide  features : {wide[feat_cols_wide].isna().sum().sum()}')
print('\n✅ Done. Both CSVs are ready for modelling.')

print('\n── Column overview (WIDE) ──')
for c in wide.columns[:10]:
    print(f'  {c}')
print(f'  ... ({len(wide.columns)} total columns)')

In [ ]:
# ── Quick sanity preview ──────────────────────────────────────────────────────
print('TIDY preview (first 10 rows):')
display(tidy.drop(columns=['col_prefix'], errors='ignore').head(10))

print('\nWIDE preview (first 3 rows, first 10 feature cols):')
preview_cols = ['subject_id','scenario','scenario_num'] + feat_cols_wide[:7]
display(wide[preview_cols].head(3))

print('\nDescriptive statistics (mav & variance, tidy):')
display(tidy[['mav','variance']].describe().round(4))